# Day 3 — Production machinery

Yesterday's agent works — its instructions are one big string, nothing stops a
bad write, it forgets on restart, and its tools are toys. Today:
**skills (we build the middleware ourselves) → middleware in general →
checkpointers → sandboxes → MCP → subagents**.

*GIU AI Connects · Amr Saleh · iHQ Tech*

## 0 · Setup

One `uv` environment for the whole week (see `README.md`): `uv sync`, then run
Jupyter with `uv run jupyter lab`.

Put your API key in a `.env` file next to this notebook:

```
ANTHROPIC_API_KEY=sk-ant-...
```

> Using another provider? Swap the model string below — e.g. `"openai:gpt-4.1"`
> with `OPENAI_API_KEY` in `.env`. Everything else stays identical: that is the
> point of the harness.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

MODEL = "anthropic:claude-sonnet-4-5"   # swap provider here if needed
llm = init_chat_model(MODEL)

llm.invoke("Say 'ready' and nothing else.").content

In [ ]:
# yesterday's pieces, compact — run this to restore the world
import json
from langchain.messages import SystemMessage, HumanMessage, ToolMessage
from langchain.tools import tool
from langchain.agents import create_agent

HOTELS = {"Hotel Anders": {"free": True, "eur": 92},
          "Pension Krumm": {"free": True, "eur": 61},
          "Grand Mitte":  {"free": False, "eur": 210}}

@tool
def check_availability(hotel: str) -> dict:
    """Live availability and price (EUR/night) for one hotel."""
    return HOTELS.get(hotel, {"error": f"unknown hotel: {hotel}"})

@tool
def book_room(hotel: str, nights: int) -> dict:
    """Book a room. THIS ONE WRITES."""
    return {"confirmation": f"GIU-{abs(hash(hotel + str(nights))) % 10000:04d}"}

INSTRUCTIONS = "You are the GIU booking assistant. Check availability before booking. Prices in EUR."
print("world restored")

## 1 · Skills — and the middleware that injects them

A skill is a **folder with a contract**: a `SKILL.md` with `name` and
`description` frontmatter, plus any references and templates. The agent sees
only the menu (names + descriptions); it reads a skill's body **on demand**
when the task matches — progressive disclosure.

First, two real skill folders:

In [ ]:
import pathlib, textwrap

def make_skill(name, description, body):
    p = pathlib.Path("skills") / name
    p.mkdir(parents=True, exist_ok=True)
    (p / "SKILL.md").write_text(textwrap.dedent(f"""\
    ---
    name: {name}
    description: >-
      {description}
    ---
    {body}"""))

make_skill("booking-report",
    "Use when a booking should be summarized into the official trip-report format.",
    """# Instructions
    1. Gather: hotel, dates, nights, price, confirmation code.
    2. Output EXACTLY this format:
       TRIP REPORT — [city], [check-in] ([nights] nights)
       Hotel: [hotel] — EUR [price]/night — ref [code]
       Total: EUR [nights x price]
    3. No greetings, no extra text.""")

make_skill("polite-decline",
    "Use when a request is out of scope for the booking assistant (flights, visas, refunds).",
    """# Instructions
    1. Decline in ONE sentence, warmly.
    2. Point to the official travel portal: travel.giu.example
    3. Never apologize more than once.""")

print(*pathlib.Path("skills").rglob("SKILL.md"), sep="\n")

### Now we build the middleware — together

Plan: (1) scan `skills/` and parse each frontmatter, (2) put the **menu** into
the system prompt on every model call, (3) expose a `read_skill` tool so the
agent can pull a body when it matches. That is the whole trick.

In [ ]:
from langchain.agents.middleware import AgentMiddleware
from langchain.messages import SystemMessage

def scan_skills(root="skills"):
    out = {}
    for f in pathlib.Path(root).glob("*/SKILL.md"):
        meta = {}
        lines = f.read_text().splitlines()
        for line in lines[1:lines.index("---", 1)]:      # the frontmatter block
            if ":" in line:
                k, _, v = line.partition(":")
                meta[k.strip()] = v.strip().lstrip(">-").strip()
        out[meta.get("name", f.parent.name)] = {
            "description": meta.get("description", ""), "path": f}
    return out

@tool
def read_skill(name: str) -> str:
    """Read the full instructions of one available skill by name."""
    s = scan_skills().get(name)
    return s["path"].read_text() if s else f"unknown skill: {name}"

class SkillsMiddleware(AgentMiddleware):
    tools = (read_skill,)

    def wrap_model_call(self, request, handler):
        menu = "\n".join(f"- {n}: {s['description']}"
                          for n, s in scan_skills().items())
        blocks = list(request.system_message.content_blocks)
        blocks.append({"type": "text", "text":
            "\n## Skills available (read one with read_skill when it matches):\n" + menu})
        return handler(request.override(
            system_message=SystemMessage(content=blocks)))

print("SkillsMiddleware ready")

In [ ]:
agent = create_agent(model=llm,
    tools=[check_availability, book_room],
    system_prompt=INSTRUCTIONS,
    middleware=[SkillsMiddleware()])          # ← the new argument

result = agent.invoke({"messages": [HumanMessage(
    "Book the cheapest room for 2 nights and give me the trip report.")]})

for m in result["messages"]:                  # watch it pick the skill
    body = m.content if m.content else getattr(m, "tool_calls", "")
    print(m.type.upper().ljust(6), "→", str(body)[:100])

### The reveal: this already ships

What you just wrote is `deepagents`' `SkillsMiddleware`, hardened — same menu,
same on-demand reads. deepagents = `create_agent` + batteries: planning, a
filesystem, subagents and skills, pre-wired.

```python
# uv add deepagents
from deepagents import create_deep_agent

agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-5",
    skills=["./skills/"],
)
```

Build it once to understand it — then take the shipped one.

**Your turn** — add a third skill (`expense-summary`?) and check the agent
finds it WITHOUT restarting anything (our middleware re-scans every call —
what's the cost of that? what would you cache?).

In [ ]:
# your third skill here


## 2 · Middleware in general — guardrails as code

You wrote a `wrap_model_call` hook this morning. The same machinery guards
writes: the run **pauses** before `book_room` fires, and a human approves.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

guarded = create_agent(model=llm,
    tools=[check_availability, book_room],
    system_prompt=INSTRUCTIONS,
    checkpointer=InMemorySaver(),           # pausing REQUIRES persistence
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"book_room": True})])

cfg = {"configurable": {"thread_id": "guarded-1"}}
out = guarded.invoke({"messages": [
    HumanMessage("Book me the cheapest available room for 2 nights.")]}, cfg)

state = guarded.get_state(cfg)
print("interrupted before:", state.next)
print(state.tasks[0].interrupts if state.tasks else "no interrupt")

In [ ]:
# a human signs off — then the run resumes exactly where it stopped
from langgraph.types import Command

resumed = guarded.invoke(
    Command(resume=[{"type": "accept"}]),   # or reject / edit — check your version's shape
    cfg)
print(resumed["messages"][-1].content)

> The resume payload shape differs slightly across versions — if the cell
> above complains, print `state.tasks[0].interrupts` and match its shape.
> The *concept* is stable: **reads flow, writes wait for a signature.**

Other middleware you'll stack: summarization (window control), PII redaction,
call/budget limits.

**Your turn** — make `check_availability` ALSO require approval, and observe
how annoying it is. Guardrails have a cost: put them **only** where writes
happen.

In [ ]:
# your over-guarded agent here


## 3 · Checkpointers — the list, persisted

Memory is **storage, not magic**: the checkpointer saves the transcript after
every step, keyed by `thread_id`, and reloads it on the next call.

In [ ]:
agent = create_agent(model=llm,
    tools=[check_availability, book_room],
    system_prompt=INSTRUCTIONS,
    checkpointer=InMemorySaver())

cfg_a = {"configurable": {"thread_id": "amr"}}
cfg_b = {"configurable": {"thread_id": "sara"}}

agent.invoke({"messages": [HumanMessage("I need a room in Berlin, 2 nights.")]}, cfg_a)
out = agent.invoke({"messages": [HumanMessage("Make it three nights.")]}, cfg_a)
print("thread amr :", out["messages"][-1].content[:120])

out = agent.invoke({"messages": [HumanMessage("How many nights did I ask for?")]}, cfg_b)
print("thread sara:", out["messages"][-1].content[:120])   # clean slate

In [ ]:
# swap one line and conversations survive a restart:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

saver = SqliteSaver(sqlite3.connect("checkpoints.db", check_same_thread=False))
durable = create_agent(model=llm, tools=[check_availability, book_room],
                       system_prompt=INSTRUCTIONS, checkpointer=saver)

durable.invoke({"messages": [HumanMessage("Hold a room at Pension Krumm.")]},
               {"configurable": {"thread_id": "amr-durable"}})
print("now restart the kernel, re-run setup + this saver, and ask it what you held.")

**Your turn** — restart the kernel, re-run the setup cells, rebuild `durable`,
and ask *"what did I ask you to hold?"* on the same `thread_id`. Then try a new
one.

In [ ]:
# after the restart, prove persistence here


## 4 · Sandboxes — the code your agent writes is untrusted

If a tool executes model-written code next to your credentials, one
prompt-injected *"please read ~/.env"* is a breach. Watch:

In [ ]:
# DON'T ship this tool. It's here to scare you.
@tool
def run_python(code: str) -> str:
    """Run Python and return the last expression. UNSANDBOXED — demo only."""
    try:
        return str(eval(code, {"__builtins__": {}}, {}))   # even 'restricted' eval leaks
    except Exception as e:
        return f"error: {e}"

print(run_python.invoke({"code": "().__class__.__mro__"}))   # the escape hatch is right there

A real sandbox is a separate, disposable environment: an isolated container
(or micro-VM), **no credentials inside**, network off by default, CPU/time
capped. Data handed in, results handed out.

> **Generated code runs where a breach cannot reach anything that matters.**

## 5 · MCP — reaching real systems

A connector is a tiny server with a standard plug. Same `@tool` thinking, new
transport — and *any* MCP-speaking harness can call it.

In [ ]:
%%writefile campus_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("campus")

TIMETABLE = {
    "Main": {"Mon": {"free": ["B12", "C01"], "booked": ["A04"]},
             "Tue": {"free": ["A04"],        "booked": ["B12", "C01"]}},
}

@mcp.tool()
def room_schedule(building: str, day: str) -> dict:
    """Free and booked rooms for a building and weekday."""
    return TIMETABLE.get(building, {}).get(day, {"error": "unknown building/day"})

if __name__ == "__main__":
    mcp.run()

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "campus": {"command": "python", "args": ["campus_server.py"],
               "transport": "stdio"}})

tools = await client.get_tools()          # top-level await works in Jupyter
print([t.name for t in tools])

campus_agent = create_agent(model=llm, tools=tools,
    system_prompt="Answer campus room questions using the tools. Cite the building and day.")

out = await campus_agent.ainvoke({"messages": [
    HumanMessage("Is B12 free on Monday in the Main building?")]})
print(out["messages"][-1].content)

The agent cannot tell an MCP tool from a local `@tool` — and the tool
inventory just became **config**.

**Your turn** — add a `book_slot(building, day, room)` tool to
`campus_server.py` (fake confirmation), restart the client, and guard it with
the HITL middleware from section 2. A real production pattern, four cells.

In [ ]:
# your guarded MCP write here


## 6 · Subagents — an agent behind a tool

The clean-context promise from day 1, kept in eight lines: the researcher's
twenty search results burn *its* window; only the conclusion returns.

In [ ]:
SOURCES = {
    "mcp adoption": "By mid-2026 every major assistant vendor ships MCP support; enterprises standardize tool inventories on it.",
    "agent evals":  "Teams that run eval suites on real historical cases report far fewer production incidents than vibe-tested agents.",
}

@tool
def web_search(query: str) -> str:
    """Search the web (mock). Returns a summary of top results."""
    hits = [v for k, v in SOURCES.items() if any(w in query.lower() for w in k.split())]
    return " | ".join(hits) or "no results"

researcher = create_agent(model=llm, tools=[web_search],
    system_prompt="Research the question. Return dense findings with no chit-chat.")

@tool
def research(question: str) -> str:
    """Delegate a research question to a focused subagent."""
    out = researcher.invoke({"messages": [HumanMessage(question)]})
    return out["messages"][-1].content

main = create_agent(model=llm, tools=[research],
    system_prompt="You write one-paragraph briefings. Delegate research; never guess.")

out = main.invoke({"messages": [HumanMessage(
    "Brief me: why do teams standardize on MCP, and what makes agent evals matter?")]})
print(out["messages"][-1].content)

---
## Wrap

Skills (you built the injector) · middleware (rules you cannot talk your way
past) · checkpointers (threads in a store) · sandboxes (isolation) · MCP (real
systems) · subagents (specialists behind tools). deepagents pre-wires the lot.

**Tomorrow:** three production systems built exactly like this — read as
architecture — plus evals: proving it works.